# 02 — Compare smoothing methods, pick best per well

For every well processed in notebook 01, plot SavGol and LOWESS side-by-side
on top of the raw points. You decide visually which one preserves the
features you care about (transitions, curvature near the freshwater lens,
no overshoot near sharp gradients), then record the choice in
`results/chosen_method.csv`.

`chosen_method.csv` is the input to notebook 03 — it tells the breakpoint
detection which smoothed file to load for each well.


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import os
if Path.cwd().name == "notebooks":
    os.chdir("..")

import pandas as pd
import matplotlib.pyplot as plt

from karst_analysis.io import parse_well_filename
from karst_analysis.sec.io import load_ysi_csv
from karst_analysis.sec.viz import plot_smoothing_comparison

print(f"CWD: {Path.cwd()}")

In [ ]:
# Folders
RAW_DIR       = Path("data/raw/sec/2022_02/D")
SAVGOL_DIR    = Path("data/processed/sec/2022_02/savgol")
LOWESS_DIR    = Path("data/processed/sec/2022_02/lowess")
COMPARE_DIR   = Path("results/figures/2022_02/comparison")
COMPARE_DIR.mkdir(parents=True, exist_ok=True)

# Match raw → savgol → lowess by well_id + date.
# Processed filenames look like  AW6D_20220215__lowess-f0.05-i2-pava.csv
def match_files(raw_path):
    info = parse_well_filename(raw_path)
    base = f"{info.well_id}_{info.date}__"
    s = sorted(SAVGOL_DIR.glob(base + "savgol-*.csv"))
    l = sorted(LOWESS_DIR.glob(base + "lowess-*.csv"))
    return info, s[-1] if s else None, l[-1] if l else None

raw_files = sorted(RAW_DIR.glob("*.csv"))
print(f"Raw files: {len(raw_files)}")

In [ ]:
# Generate comparison figures + interactive matplotlib viewer
choices = []  # list of dicts to be saved at the end

for raw_path in raw_files:
    info, sav_path, low_path = match_files(raw_path)
    if sav_path is None or low_path is None:
        print(f"⚠️  Missing processed file for {info.well_id}; run notebook 01 first.")
        continue

    df_raw = load_ysi_csv(raw_path)
    df_sav = pd.read_csv(sav_path)
    df_low = pd.read_csv(low_path)

    # Side-by-side comparison PNG
    fig_path = COMPARE_DIR / f"{info.well_id}_{info.date}__comparison.png"
    plot_smoothing_comparison(
        df_raw["depth_m"].to_numpy(),  df_raw["sec_uS_cm"].to_numpy(),
        df_sav["depth_m"].to_numpy(),  df_sav["sec_uS_cm"].to_numpy(),
        df_low["depth_m"].to_numpy(),  df_low["sec_uS_cm"].to_numpy(),
        output_path=fig_path,
        title=f"{info.well_id} {info.date} — SavGol vs LOWESS",
    )

    print(f"\n=== {info.well_id} {info.date} ===")
    print(f"  SavGol: {sav_path.name}")
    print(f"  LOWESS: {low_path.name}")
    print(f"  Figure: {fig_path}")

    # Inline display
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(df_raw["sec_uS_cm"], df_raw["depth_m"],
               s=4, color="lightgray", alpha=0.5, label="Raw")
    ax.plot(df_sav["sec_uS_cm"], df_sav["depth_m"],
            color="#c0392b", lw=1.4, label="SavGol")
    ax.plot(df_low["sec_uS_cm"], df_low["depth_m"],
            color="#1f4e79", lw=1.4, label="LOWESS")
    ax.invert_yaxis()
    ax.set_xlabel("SEC")
    ax.set_ylabel("Depth [m]")
    ax.set_title(f"{info.well_id} {info.date}")
    ax.legend()
    ax.grid(True, ls=":", alpha=0.5)
    plt.show()

## Record your choices

Edit the dictionary below to record which method you picked for each well.
Then run the cell to save `results/chosen_method.csv`.


In [ ]:
# >>> EDIT THIS <<<
# Format: well_id -> "savgol" or "lowess"
chosen_method = {
    "AW6D":   "lowess",   # placeholder
    "AW5D":   "lowess",   # placeholder
    "BW3D":   "lowess",   # placeholder
    "LRS69D": "lowess",   # placeholder
    "LRS70D": "lowess",   # placeholder
}

# Save (paths to the chosen file, for downstream notebooks)
rows = []
for raw_path in raw_files:
    info, sav_path, low_path = match_files(raw_path)
    if info.well_id not in chosen_method:
        continue
    method = chosen_method[info.well_id]
    chosen_file = sav_path if method == "savgol" else low_path
    rows.append({
        "well_id": info.well_id,
        "date": info.date,
        "method": method,
        "chosen_file": str(chosen_file),
    })

chosen_df = pd.DataFrame(rows)
out_path = Path("results/chosen_method.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
chosen_df.to_csv(out_path, index=False)
print(f"Saved {len(chosen_df)} choices to {out_path}")
chosen_df